# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema at the URL below. It contains structured records on clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata and records
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object
metadata = dataset.metadata

# Display dataset overview
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This will help us understand the schema and structure of the dataset.


In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.record_sets

print("Available record sets in the dataset:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'No description'}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - Field name: {f.name} @id: {f.id} Data type: {f.data_type}")
    print()

# For demonstration, preview one record from each record set
for rs in record_sets:
    print(f"First record from RecordSet '@id': {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    if records:
        print(records[0])
    else:
        print("No records found.")
    print("---")

## 3. Data Extraction
Load data from each RecordSet into a DataFrame for analysis. All entities are referenced by their `@id` fields.


In [ ]:
# Prepare dataframes from all record sets
dataframes = {}

record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print columns and preview head for the first record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Columns in RecordSet '@id': {first_record_set_id}")
    print(dataframes[first_record_set_id].columns.tolist())
    print(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalization, and grouping. In this example, we will select a numeric field, filter by a threshold, normalize its values, and group the records. All fields referenced by their `@id`.


In [ ]:
# Example EDA on the first record set

# Choose record_set_id and numeric_field (by @id)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Find a numeric field by @id
rs_obj = next((rs for rs in dataset.record_sets if rs.id == record_set_id), None)
numeric_fields = [field for field in rs_obj.fields if field.data_type in ('Float', 'Integer', 'Number')]

if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    # Use a threshold for demonstration
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field (if available)
        group_fields = [field for field in rs_obj.fields if field.data_type == 'Text']
        if group_fields:
            group_field_id = group_fields[0].id
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped data by {group_field_id}:")
                print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset. All references by `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric distributions for first record set (example)
if numeric_fields and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a group_field, show boxplot
    if group_fields and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze clinical, hormonal, and genetic records from a Croissant FAIR² dataset using `mlcroissant`. All entities were referenced by their `@id`. The techniques used here can be extended for more sophisticated analyses and visualizations across multiple record sets and fields.
